Installazione dipendenze

**CONSIGLIO:** evita assolutamente di far girare questo codice su Windows (face-recognition ha troppi problemi di dipendenze)



In [1]:
!pip install setuptools
!pip install dlib
!pip install opencv-python
!pip install Numpy
!pip install face-recognition
!pip install face-recognition-models
!pip install gc-python-utils
!pip install shutils
!pip install insightface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=a188ce5b8642215cde4b252aef86edb0e573271281ae3e8cb7d68f2ff1eb69d8
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models
  Preparing metadata (setup.py) ... done
  Created wheel for gc-python-utils: filename=gc_python_utils-0.0.1-py3-none-any.whl size=1299 sha256=6ea4cf3bda9ddd3b9fc222c1d05200e3bf8a4223a55fc13195aea2e60b06d92b
  Stored in directory: /root/.cache/pip/wheels/c4/7f/ec/7f462698d7e015ef9f5237f1c3b3b04da533da1a9635db593f
Successfully built gc-python-utils
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.4 MB/s eta 0:00:00
  Created wheel for shutils: filename=shutils-0.1.0-py3-none-any.wh

Accesso a drive condiviso in cui ci sono i dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Parte di pulizia del dataset:

estrattore del volto principale nel video e degli encoding ad esso associati

In [ ]:
import cv2
import face_recognition
from collections import Counter
import numpy as np

class AutoReferenceExtractor:

    def __init__(self):
        pass


    def extract_main_face_from_video(self, video_path, sample_interval=10, resize_scale=0.5):
        # sample_interval si può estendere per avere maggiore efficienza

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # CAP_PROP_FRAME_COUNT retrieves the total number of frames in a video file

        face_counter = []
        face_images = []  # salva un'immagine di esempio per ogni volto unico
        known_face_encodings = []
        encodings_per_person = []

        frame_idx = 0
        print(f"Analisi {total_frames} frame per identificare il soggetto principale...")

        processed_frames = 0
        for i in range(0, total_frames):
            ret, frame = cap.read()    # cap.read() returns a bool (true if the frame has been read properly) and the frame itself
            if not ret:
                break


            # if inserito qualora si volessero analizzare un sottoinsieme dei frame
            if frame_idx % sample_interval == 0:
                processed_frames += 1
                small_frame = cv2.resize(frame, (0, 0), fx=resize_scale, fy=resize_scale)
                rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

                #nome_file = f"frame_{i:02d}.jpg"
                #cv2.imwrite(nome_file, rgb)
                #print("Frame saved successfully!")

                # find faces
                face_locations = face_recognition.face_locations(rgb_small_frame)
                # detects the presence and locations of human faces in a given image; it takes a numpy array representing an image in RGB color format and returns a list of tuples, where each tuple contains the coordinates of a detected face


                face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
                # converts the facial features of detected faces in an image into a compact numerical representation. It generates a 128-dimensional vector (a list of 128 numbers) for each face, which acts as a unique signature for that person.


                #if (len(face_encodings)>0):
                    #print("Abbiamo ", {len(face_encodings)}, " facce")
                    #print(face_encodings[0])



                for encoding in face_encodings:
                    trovato = False

                    for index, known_encoding in enumerate(known_face_encodings):
                        result = face_recognition.compare_faces([known_encoding], encoding, tolerance=0.6)
                        #Studi su embedding mostrano che testi casuali e non correlati tendono ad avere cosine similarity mediamente intorno a 0.34-0.35 . Questo significa che 0.6 è già abbastanza sopra la "media del rumore di fondo". Dai testi, il concetto è stato trasportato alle immagini

                        # risultato è una lista di bool anche se ha size 1
                        if result[0]:
                            # Se la persona esiste già, aggiorna magari l'immagine di riferimento (opzionale)
                            face_images[index] = frame
                            trovato = True
                            encodings_per_person[index].append(encoding)
                            face_counter[index] += 1
                            break # Esci dal ciclo: abbiamo trovato chi è

                    # 2. Se NON è stato trovato, aggiungilo come nuova persona
                    if not trovato:
                        known_face_encodings.append(encoding) # Salva l'encoding per i prossimi confronti
                        face_images.append(frame)             # Salva il frame corrispondente
                        face_counter.append(1)
                        encodings_per_person.append([encoding])
                        print(f"Nuova persona rilevata! Totale: {len(known_face_encodings)}")

            frame_idx += 1



        cap.release()
        known_face_encodings = None
        face_locations = None
        face_encodings = None

        if not face_counter:
            return None, None
            #raise ValueError("Nessun volto trovato nel video")

        # Trova il volto più frequente (soggetto principale)
        idx = face_counter.index(max(face_counter))

        print(f"\nSoggetto principale identificato!")
        print(f"Apparso in {face_counter[idx]} frame su {processed_frames} analizzati")

        return face_images[idx], encodings_per_person[idx]


oggetto che si occupa di eliminare i frame privi del soggetto principale e di formare con essi un nuovo video da salvare

In [ ]:
import cv2
import face_recognition
import numpy as np

class VideoFaceFilter:

    def __init__(self, main_encodings, tolerance=0.6, resize_scale=0.5):
        self.main_encodings = main_encodings
        self.tolerance = tolerance
        self.resize_scale = resize_scale


    def frame_contains_subject(self, frame):
        # Resize per velocità
        small = cv2.resize(frame, (0, 0), fx=self.resize_scale, fy=self.resize_scale)
        rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

        # Detect
        face_locations = face_recognition.face_locations(rgb)
        if not face_locations:
            return False

        # Encode
        encodings = face_recognition.face_encodings(rgb, face_locations)

        # Check match
        for enc in encodings:
            distances = face_recognition.face_distance(self.main_encodings, enc)

            # 👇 usa la distanza minima
            if np.min(distances) < self.tolerance:
                return True

        return False


    def filter_video(self, video_path, sample_interval=1):
        cap = cv2.VideoCapture(video_path)
        kept_frames = []
        total = 0
        kept = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if total % sample_interval == 0:
                if self.frame_contains_subject(frame):
                    kept_frames.append(frame)
                    kept += 1

            total += 1

        cap.release()

        print(f"Frame totali analizzati: {total}")
        print(f"Frame mantenuti: {kept}")

        return kept_frames



    def frames_to_video(self, frames, output_path, fps=25):
        if len(frames) == 0:
            raise ValueError("No frames")

        h, w, _ = frames[0].shape

        # forza consistenza dimensioni
        frames = [cv2.resize(f, (w, h)) for f in frames]

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # prova H264

        out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

        if not out.isOpened():
            raise RuntimeError("VideoWriter non aperto: codec non supportato")

        for f in frames:
            out.write(f)

        out.release()

main che sfrutta i due oggetti precedentemente definiti

In [ ]:
#from reference_extractor import AutoReferenceExtractor
import cv2
import os
from pathlib import Path
import gc
#from video_filter import VideoFaceFilter


# Utilizzo
extractor = AutoReferenceExtractor()
#reference_image, reference_encodings = extractor.extract_main_face_from_video("/content/drive/MyDrive/YW_WILTY_EP70_truth16.mp4")

# Salva la reference image per future analisi
#cv2.imwrite("reference_face.jpg", reference_image)

#videoFilter = VideoFaceFilter(reference_encodings)

#new_frames = videoFilter.filter_video("/content/drive/MyDrive/YW_WILTY_EP70_truth16.mp4")

#videoFilter.frames_to_video(new_frames, "VIDEO.mp4")


root = "/content/drive/MyDrive/FVAB 2025-2026"

file_path = "/content/lista_video.txt"  # percorso del file .txt

with open(file_path, "r") as f:
    video_list = [line.strip() for line in f]

#for path, dirs, files in os.walk(root):
#    for file in files:
#        if file.endswith(".mp4"):
#            print(os.path.join(path, file))
#            video_list.append(os.path.join(path, file))

print(len(video_list))

subjectNotDetected = []
for path in video_list:
# il giorno 22/04 mi ero fermato all'elemento 831
  if "Bag of Lies" in path or "Real Life Trial" in path:
    continue

  print(f"Sto analizzando {path}")
  reference_image, reference_encodings = extractor.extract_main_face_from_video(path)

  if (reference_image is None):
    subjectNotDetected.append(path)
    continue

  # Salva la reference image per future analisi
  cv2.imwrite("reference_face.jpg", reference_image)

  videoFilter = VideoFaceFilter(reference_encodings)

  dataset_name = ""
  if "DOLOS" in str(path):
    dataset_name = "DOLOS"
  elif "Real Life Trial" in str(path):
    dataset_name = "REAL_LIFE_TRIAL"
  else:
    dataset_name = "BAG_OF_LIES"

  new_frames = videoFilter.filter_video(path)

  if "Truth" in str(path) or "Truthful" in str(path):
    word = "TRUTHS"
  elif "DOLOS/Lies" in str(path) or "Deceptive" in str(path):
    word = "LIES"
  else:
    parts = Path(path).parts

    # trova la prima parte che inizia con "User"
    idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
    word = parts[idx] + "/" + parts[idx+1]

  # se path non essite lo crei e poi salvi
  output_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{os.path.basename(path)}")
  output_path.parent.mkdir(parents=True, exist_ok=True)
  videoFilter.frames_to_video(new_frames, str(output_path))

  print(f"Pulito il video {os.path.basename(path)}\n\n")
  gc.collect()


print("Abbiamo avuto problemi con i seguenti video:")
print(subjectNotDetected)


FileNotFoundError: [Errno 2] No such file or directory: '/content/lista_video.txt'

individuazione dei video non puliti in quanto considerati privi di volti

In [ ]:
import os

root = "/content/drive/MyDrive/DATASET_FVAB"

file_path = "/content/lista_video.txt"  # percorso del file .txt



with open(file_path, "r") as f:
    video_list = [line.strip() for line in f]

cleaned_video_list = []
not_cleaned_video_list = []
for root, dirs, files in os.walk(root):
    for file in files:
        if file.endswith(".mp4"):
            #print(os.path.join(path, file))
            cleaned_video_list.append(os.path.join(root, file))


#print(len(video_list))
#print(cleaned_video_list)

counter = 0
for video in video_list:
    if "DOLOS" in video:
        counter += 1
        flag = False
        for cleaned_video in cleaned_video_list:
            if os.path.basename(video).lower() == os.path.basename(cleaned_video).lower():
                flag = True
                break

        if not flag:
            not_cleaned_video_list.append(video)

print(not_cleaned_video_list)
print(counter)
print(len(cleaned_video_list))

# Salva la lista in un file di testo standard
with open("video_non_puliti.txt", "w", encoding="utf-8") as f:
    for video in not_cleaned_video_list:
        f.write(f"{video}\n")

print(f"Salvataggio completato: {len(not_cleaned_video_list)} video scritti nel file.")


# si sono persi 27 video (1675 video totali)

['/content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Lies/SB_WILTY_EP32_lie2.mp4', '/content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Lies/SB_WILTY_EP32_lie3.mp4', '/content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Lies/LS_WILTY_EP8_lie14.mp4', '/content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Truth/AN_WILTY_EP18_truth5.mp4']
1648
1644
Salvataggio completato: 4 video scritti nel file.


cambio parametri per i 4 video non presi di DOLOS

In [ ]:
#from reference_extractor import AutoReferenceExtractor
import cv2
import os
from pathlib import Path
import gc
#from video_filter import VideoFaceFilter


# Utilizzo
extractor = AutoReferenceExtractor()

root = "/content/drive/MyDrive/FVAB 2025-2026"

file_path = "/content/video_non_puliti.txt"  # percorso del file .txt

with open(file_path, "r") as f:
    video_list = [line.strip() for line in f]

subjectNotDetected = []
for path in video_list:
  print(f"Sto analizzando {path}")
  reference_image, reference_encodings = extractor.extract_main_face_from_video(path, resize_scale=0.75)

  if (reference_image is None):
    subjectNotDetected.append(path)
    continue

  # Salva la reference image per future analisi
  cv2.imwrite("reference_face.jpg", reference_image)

  videoFilter = VideoFaceFilter(reference_encodings, resize_scale=0.75)

  dataset_name = ""
  if "DOLOS" in str(path):
    dataset_name = "DOLOS"
  elif "Real Life Trial" in str(path):
    dataset_name = "REAL_LIFE_TRIAL"
  else:
    dataset_name = "BAG_OF_LIES"

  new_frames = videoFilter.filter_video(path)

  if "Truth" in str(path) or "Truthful" in str(path):
    word = "TRUTHS"
  elif "DOLOS/Lies" in str(path) or "Deceptive" in str(path):
    word = "LIES"
  else:
    parts = Path(path).parts

    # trova la prima parte che inizia con "User"
    idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
    word = parts[idx] + "/" + parts[idx+1]

  # se path non essite lo crei e poi salvi
  output_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{os.path.basename(path)}")
  output_path.parent.mkdir(parents=True, exist_ok=True)
  videoFilter.frames_to_video(new_frames, str(output_path))

  print(f"Pulito il video {os.path.basename(path)}\n\n")
  gc.collect()


print("Abbiamo avuto problemi con i seguenti video:")
print(subjectNotDetected)

# Salva la lista in un file di testo standard
with open("video_non_puliti2.txt", "w", encoding="utf-8") as f:
    for video in subjectNotDetected:
        f.write(f"{video}\n")

print(f"Salvataggio completato: {len(subjectNotDetected)} video scritti nel file.")


Sto analizzando /content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Lies/SB_WILTY_EP32_lie2.mp4
Analisi 125 frame per identificare il soggetto principale...
Sto analizzando /content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Lies/SB_WILTY_EP32_lie3.mp4
Analisi 125 frame per identificare il soggetto principale...
Sto analizzando /content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Lies/LS_WILTY_EP8_lie14.mp4
Analisi 75 frame per identificare il soggetto principale...
Nuova persona rilevata! Totale: 1
Nuova persona rilevata! Totale: 2
Nuova persona rilevata! Totale: 3

Soggetto principale identificato!
Apparso in 3 frame su 8 analizzati
Frame totali analizzati: 75
Frame mantenuti: 30
Pulito il video LS_WILTY_EP8_lie14.mp4


Sto analizzando /content/drive/MyDrive/FVAB 2025-2026/Datasets/DOLOS/Truth/AN_WILTY_EP18_truth5.mp4
Analisi 75 frame per identificare il soggetto principale...
Nuova persona rilevata! Totale: 1
Nuova persona rilevata! Totale: 2

Soggetto principale identificato!
App

I restanti due video non verranno integrati nel dataset (da scrivere nel report)

Analisi numero frame per ciascun video dei restanti due dataset

In [ ]:
#from reference_extractor import AutoReferenceExtractor
import cv2
import os
from pathlib import Path
import gc
#from video_filter import VideoFaceFilter


root = "/content/drive/MyDrive/FVAB 2025-2026"

file_path = "/content/lista_video.txt"  # percorso del file .txt

video_list = []

with open(file_path, "r") as f:
    for row in f:
        row = row.strip()
        if "Real Life Trial" in row or "Bag of Lies" in row:
          video_list.append(row)

for video in video_list:
  cap = cv2.VideoCapture(video)
  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
  print(f"{video}\nNumero frame: {total_frames}")
  cap.release()

/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_0/video.mp4
Numero frame: 245
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_2/video.mp4
Numero frame: 231
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_1/video.mp4
Numero frame: 211
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_3/video.mp4
Numero frame: 210
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_4/video.mp4
Numero frame: 167
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_5/video.mp4
Numero frame: 220
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_0/video.mp4
Numero frame: 304
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_1/video.mp4
Numero frame: 234
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_2/video.mp4
Numero frame: 218
/content/d

'\nfor video in video_list:\n  print(video)\n#print(video_list)\n\nsubjectNotDetected = []\ntooLong = []\nfor path in video_list:\n  print(f"Sto analizzando {path}")\n  reference_image, reference_encodings = extractor.extract_main_face_from_video(path, resize_scale=0.5)\n\n  if (reference_image is None):\n    subjectNotDetected.append(path)\n    continue\n\n  # Salva la reference image per future analisi\n  cv2.imwrite("reference_face.jpg", reference_image)\n\n  if len(reference_encodings) > 22:\n    tooLong.append(path)\n  else:\n    videoFilter = VideoFaceFilter(reference_encodings, resize_scale=0.5)\n\n\n    dataset_name = "REAL_LIFE_TRIAL"\n\n    new_frames = videoFilter.filter_video(path)\n\n    if "Truth" in str(path) or "Truthful" in str(path):\n      word = "TRUTHS"\n    elif "DOLOS/Lies" in str(path) or "Deceptive" in str(path):\n      word = "LIES"\n    else:\n      parts = Path(path).parts\n      # trova la prima parte che inizia con "User"\n      idx = next((i for i, p in e

Sotto i 200 frame (compreso), applichiamo procedura standard

In [ ]:
#from reference_extractor import AutoReferenceExtractor
import cv2
import os
from pathlib import Path
import gc
#from video_filter import VideoFaceFilter


extractor = AutoReferenceExtractor()

root = "/content/drive/MyDrive/FVAB 2025-2026"

file_path = "/content/lista_video.txt"  # percorso del file .txt

video_list = []
good_list = []
bad_list = []


with open(file_path, "r") as f:
    for row in f:
        row = row.strip()
        if "Real Life Trial" in row or "Bag of Lies" in row:
          video_list.append(row)

for video in video_list:
  cap = cv2.VideoCapture(video)
  total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
  #print(f"{video}\nNumero frame: {total_frames}")
  cap.release()
  if total_frames <= 200:
    good_list.append(video)
  else:
    bad_list.append(video)


 # Salva la lista in un file di testo standard
with open("video_non_puliti_bag_trial.txt", "w", encoding="utf-8") as f:
    for video in bad_list:
        f.write(f"{video}\n")

print(f"Salvataggio completato: {len(bad_list)} video scritti nel file.")


for video in good_list:
  print(video)
#print(video_list)

subjectNotDetected = []
tooLong = []
for path in good_list:
  print(f"Sto analizzando {path}")
  reference_image, reference_encodings = extractor.extract_main_face_from_video(path, resize_scale=0.25)

  if (reference_image is None):
    subjectNotDetected.append(path)
    continue

  # Salva la reference image per future analisi
  cv2.imwrite("reference_face.jpg", reference_image)


  videoFilter = VideoFaceFilter(reference_encodings, resize_scale=0.25)

  if "Real Life Trial" in path:
    dataset_name = "REAL_LIFE_TRIAL"
  else:
    dataset_name = "BAG_OF_LIES"

  new_frames = videoFilter.filter_video(path)

  if "Truth" in str(path) or "Truthful" in str(path):
    word = "TRUTHS"
  elif "Deceptive" in str(path):
    word = "LIES"
  else:
    parts = Path(path).parts
    # trova la prima parte che inizia con "User"
    idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
    word = parts[idx] + "/" + parts[idx+1]

  # se path non esiste lo crei e poi salvi
  output_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{os.path.basename(path)}")
  output_path.parent.mkdir(parents=True, exist_ok=True)
  videoFilter.frames_to_video(new_frames, str(output_path))

  print(f"Pulito il video {os.path.basename(path)}\n\n")


print("Nessun soggetto nei seguenti video:")
print(subjectNotDetected)


# Salva la lista in un file di testo standard
with open("soggetto_assente_bag_trial.txt", "w", encoding="utf-8") as f:
    for video in subjectNotDetected:
        f.write(f"{video}\n")

print(f"Salvataggio completato: {len(subjectNotDetected)} video scritti nel file.")



Salvataggio completato: 392 video scritti nel file.
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_4/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_2/run_1/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_3/run_0/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_3/run_2/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_5/run_1/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_5/run_3/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_5/run_4/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_5/run_5/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_5/run_7/video.mp4
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_5/run_6/video.mp4
/content/drive/MyDrive/FVAB 

Cambio di metodologia: analizziamo i frame (non i video), li salviamo nel nuovo dataset e poi successivamente a partire da essi creiamo il video "pulito"

In [ ]:
import cv2
import os
from pathlib import Path
import gc
import shutil

root = "/content/drive/MyDrive/FVAB 2025-2026"

video_list = []

file_path = "/content/video_non_puliti_bag_trial.txt"
with open(file_path, "r") as f:
    for row in f:
        row = row.strip()
        if "Real Life Trial" in row or "Bag of Lies" in row:
          video_list.append(row)

video_list.sort()
print(len(video_list))

subjectNotDetected = []
video_path_list = []


extractor = AutoReferenceExtractor()
# fatti i primi 96 video
for video in video_list[164:]:
      print(f"Sto analizzando {video}")
      reference_image, reference_encodings = extractor.extract_main_face_from_video(video, resize_scale=0.25)

      if (reference_image is None):
            subjectNotDetected.append(video)
            continue

      parts = Path(video).parts

      num = None
      if "Real Life Trial" in video:
            num = parts[9].split('.')[0]
            num = int(num.split('_')[-1])
            frames_root = Path(video).parent.parent.parent / f"extracted_frames/{parts[8]}/{num}"
      else:
            frames_root =  Path(video).parent.parent.parent.parent / f"extracted_frames/{parts[8]}/{parts[9]}"

      print(frames_root)


      if "Real Life Trial" in video:
            dataset_name = "REAL_LIFE_TRIAL"
      else:
            dataset_name = "BAG_OF_LIES"

      if "Truth" in str(video) or "Truthful" in str(video):
            word = f"TRUTHS/{num}"
      elif "Deceptive" in str(video):
            word = f"LIES/{num}"
      else:
            # trova la prima parte che inizia con "User"
            idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
            word = parts[idx] + "/" + parts[idx+1]


      filter = VideoFaceFilter(reference_encodings)

      output_path = None

      for path, dirs, files in os.walk(frames_root):
            for file in files:
                  if not file.lower().endswith(".jpg"):
                        continue

                  frame_path = os.path.join(path, file)
                  frame = cv2.imread(frame_path)

                  if frame is None:
                        continue

                  if filter.frame_contains_subject(frame):
                        output_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{os.path.basename(Path(frame_path))}")
                        output_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(output_path), frame)
                        #print(output_path)

                  if output_path is not None and "00.jpg" in output_path.name:
                        print(f"Salvato frame {os.path.basename(output_path)}\n\n")

      if output_path is not None:
            print("Salvataggio dei frame terminato...")
            video_path_list.append(str(output_path.parent))
            print(f"\nVideo finora salvati:")
            for video_path in video_path_list:
                  print(f"{video_path}")
            print("")

      del filter

print("Nessun soggetto nei seguenti video:")
print(subjectNotDetected)

# Salva la lista in un file di testo standard
with open("soggetto_assente_bag_trial.txt", "w", encoding="utf-8") as f:
      for video in subjectNotDetected:
            f.write(f"{video}\n")

print(f"Salvataggio completato: {len(subjectNotDetected)} video scritti nel file.")



248
Sto analizzando /content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Deceptive/trial_lie_019.mp4
Analisi 1112 frame per identificare il soggetto principale...
Nuova persona rilevata! Totale: 1

Soggetto principale identificato!
Apparso in 111 frame su 111 analizzati
/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/extracted_frames/Deceptive/19
Salvato frame frame-0200.jpg


Salvato frame frame-0300.jpg


Salvato frame frame-0400.jpg


Salvato frame frame-0500.jpg


Salvato frame frame-0600.jpg


Salvato frame frame-0700.jpg


Salvato frame frame-0800.jpg


Salvato frame frame-0900.jpg


Salvato frame frame-1000.jpg


Salvato frame frame-1100.jpg


Salvato frame frame-0100.jpg


Salvataggio dei frame terminato...

Video finora salvati:
/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/19

Sto analizzando /content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Deceptive/trial_lie_020.mp4
Analisi 363 frame per identificare il soggetto 

qui si effettua il passaggio da frame "cleaned" a video

In [ ]:
import cv2
import os
from glob import glob
from pathlib import Path

root = "/content/drive/MyDrive/DATASET_FVAB"


# Lettura
video_list = []
with open("original_info.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))

frame_path_list = []

for path, dirs, files in os.walk(root):
      if any(file.lower().endswith('.jpg') for file in files):
            frame_path_list.append(path)




for p in frame_path_list:
      images = sorted(glob(os.path.join(p, "*.jpg")))

      # Lettura primo frame per ottenere le dimensioni
      frame = cv2.imread(images[0])
      height, width, _ = frame.shape

      fourcc = cv2.VideoWriter_fourcc(*"mp4v")

      video_path = ""
      fps = None
      parts = Path(p).parts
      if "DOLOS" in p:
            continu
      if "BAG_OF_LIES" in p:
            video_path = f"{p}/video.mp4"

            for v in video_list:
                  if f"{parts[-2]}/{parts[-1]}" in v[0]:
                        fps = v[2]
                        break
      else:
            parts = Path(p).parts
            video_path = f"{Path(p).parent}/{parts[-1]}.mp4"
            for v in video_list:
                  if f"{parts[-1]}" in v[0]:
                        if ("TRUTHS" in p and "Truthful" in v[0]) or ("LIES" in p and "Deceptive" in v[0]):
                              fps = v[2]
                              break


      video = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

      for img_path in images:
            frame = cv2.imread(img_path)
            if frame is None:
                  continue
            video.write(frame)

      video.release()
      print(f"Salvato video in {video_path}")




parte di analisi della situazione per capire se il modello usato ha funzionato su tutti i video e quanti ne ha scartati (falsi negativi)

In [ ]:
import cv2
import os
from pathlib import Path
import gc


root = "/content/drive/MyDrive/FVAB 2025-2026"

video_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if file.endswith(".mp4"):
                  video_path = os.path.join(path, file)
                  video = cv2.VideoCapture(video_path)
                  nFrame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                  fps = int(video.get(cv2.CAP_PROP_FPS))
                  video.release()
                  video_list.append((video_path, nFrame, fps))


# Scrittura
with open("original_info.txt", "w") as f:
      for v in video_list:
            f.write(f"{v[0]},{v[1]},{v[2]}\n")

# Lettura
video_list = []
with open("original_info.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))


for v in video_list:
      print(v)

('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_0/video.mp4', 245, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_2/video.mp4', 231, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_1/video.mp4', 211, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_3/video.mp4', 210, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_4/video.mp4', 167, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_5/video.mp4', 220, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_0/video.mp4', 304, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_1/video.mp4', 234, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_2/video.mp4', 218, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of

analisi video generati con pulizia

In [ ]:
import cv2
import os
from pathlib import Path
import gc


root = "/content/drive/MyDrive/DATASET_FVAB"

video_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if file.endswith(".mp4"):
                  video_path = os.path.join(path, file)
                  video = cv2.VideoCapture(video_path)
                  nFrame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                  fps = int(video.get(cv2.CAP_PROP_FPS))
                  video.release()
                  video_list.append((video_path, nFrame, fps))


# Scrittura
with open("cleaned_info.txt", "w") as f:
      for v in video_list:
            f.write(f"{v[0]},{v[1]},{v[2]}\n")

# Lettura
video_list = []
with open("cleaned_info.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))


for v in video_list:
      print(v)

('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP16_lie4.mp4', 89, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP15_lie16.mp4', 119, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP17_lie17.mp4', 63, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP16_lie7.mp4', 75, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP22_lie28.mp4', 185, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP16_lie19.mp4', 100, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP19_lie14.mp4', 59, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP18_lie9.mp4', 150, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP21_lie9.mp4', 53, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP15_lie5.mp4', 87, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP22_lie5.mp4', 75, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP24_lie2.mp4', 250, 25)
('/con

Tiriamo le somme: quali video sono stati inficiati dalla pulizia, per quanti frame e quali invece non sono stati puliti perché già buoni o perché ritenuti privi di volti

In [ ]:
import cv2
import os
from pathlib import Path
import gc

# Lettura
original_video_list = []
with open("original_info.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            original_video_list.append((path, int(nFrame), int(fps)))


# Lettura
cleaned_video_list = []
with open("cleaned_info.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            cleaned_video_list.append((path, int(nFrame), int(fps)))

print(len(original_video_list))
print(len(cleaned_video_list))


original_video_list_changed = []

ctr_bof = 0
ctr_rlt = 0
ctr_dol = 0
for v in original_video_list:

      dataset_name = None
      word = None
      parts = Path(v[0]).parts
      filename = parts[-1]

      if "Real Life Trial" in v[0]:
            dataset_name = "REAL_LIFE_TRIAL"
            ctr_rlt += 1
      elif "DOLOS" in v[0]:
            dataset_name = "DOLOS"
            ctr_dol += 1
      else:
            dataset_name = "BAG_OF_LIES"
            ctr_bof += 1


      if "Truth" in str(v[0]) or "Truthful" in str(v[0]):
            word = "TRUTHS"
      elif "Deceptive" in str(v[0]) or "/Lies" in str(v[0]): # necessario "/Lies" perché Lies presente anche in "Bag of Lies"
            word = "LIES"
      else:
            # trova la prima parte che inizia con "User"
            idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
            word = parts[idx] + "/" + parts[idx+1]


      if "Real Life" in str(v[0]):
            filename = f"{int(parts[-1].split("_")[2].split(".")[0])}.mp4"

      # se path non esiste lo crei e poi salvi
      new_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{filename}")


      record = (str(new_path), v[1], v[2])
      original_video_list_changed.append(record)


not_analyzed_list = []
less_frames_list = []
for o in original_video_list_changed:
      flag = False
      for c in cleaned_video_list:
            if o[0] == c[0]:
                  flag = True

                  if c[1] < o[1]:
                        less_frames_list.append(o)
                        less_frames_list.append(c)

      if not flag:
            not_analyzed_list.append(o)

counter_bof = 0
counter_rlt = 0
print(f"\nVideo non analizzati: {len(not_analyzed_list)}")
for v in not_analyzed_list:
      print(v)
      if "BAG_OF_LIES" in v[0]:
            counter_bof += 1
      elif "REAL_LIFE_TRIAL" in v[0]:
            counter_rlt += 1

print(f"\nVideo non analizzati di Bag of Lies: {counter_bof} su {ctr_bof}\nVideo non analizzati di Real Life Trial: {counter_rlt} su {ctr_rlt}")

print(f"\nVideo che hanno subito riduzione di frame: {int(len(less_frames_list)/2)}")
counter = 0
for v in less_frames_list:
      if counter % 2 == 0:
            print("")
      print(v)
      counter += 1 % 2


2094
1911

Video non analizzati: 183
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_0/video.mp4', 276, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_1/video.mp4', 242, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_10/video.mp4', 225, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_2/video.mp4', 309, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_8/video.mp4', 224, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7/run_0/video.mp4', 327, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7/run_1/video.mp4', 277, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7/run_4/video.mp4', 336, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8/run_8/video.mp4', 242, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8/run_2/video.mp4', 308, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8/run_3/video.mp4', 315, 30)
('/content/drive/MyDrive/DATASET_FVAB/

Testiamo nuovo modello di rilevamento facciale

In [3]:
!pip install onnxruntime
!pip install onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.3 MB/s eta 0:00:00


Definizione nuovo oggetto che usa modello differente per estrarre volto principale ed encoding ad esso associati

In [4]:
class AutoReferenceExtractorInsightFace:

    def __init__(self, det_size=(640, 640), similarity_threshold=0.6):
        self.app = FaceAnalysis(
            name='buffalo_l',
            providers=['CUDAExecutionProvider']
        )
        self.app.prepare(ctx_id=0, det_size=det_size)
        self.similarity_threshold = similarity_threshold


    def extract_main_face_from_video(self, video_path, sample_interval=10, resize_scale=0.5):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        face_counter = []
        face_frames = []  # LISTA DI LISTE: ogni persona ha la sua lista di frame
        known_face_encodings = []
        encodings_per_person = []

        frame_idx = 0
        processed_frames = 0

        print(f"Analisi {total_frames} frame per identificare il soggetto principale...")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sample_interval == 0:
                if resize_scale != 1.0:
                    small_frame = cv2.resize(frame, (0, 0), fx=resize_scale, fy=resize_scale)
                else:
                    small_frame = frame

                faces = self.app.get(small_frame)

                for face in faces:
                    encoding = face.embedding
                    trovato = False

                    for index, known_encoding in enumerate(known_face_encodings):
                        similarity = np.dot(known_encoding, encoding)

                        if similarity > self.similarity_threshold:
                            face_frames[index].append(frame.copy())
                            encodings_per_person[index].append(encoding)
                            face_counter[index] += 1
                            trovato = True
                            break

                    if not trovato:
                        known_face_encodings.append(encoding)
                        face_frames.append([frame.copy()])  # Nuova lista per nuova persona
                        face_counter.append(1)
                        encodings_per_person.append([encoding])
                        print(f"Nuova persona rilevata! Totale: {len(known_face_encodings)}")

                if processed_frames % 10 == 0:
                    print(f"  Processati {processed_frames+1} frame su {int(total_frames/sample_interval)+1}...")

                processed_frames += 1
            frame_idx += 1

        cap.release()

        if not face_counter:
            print("Nessun volto trovato nel video!")
            return None, None

        # Trova il soggetto principale
        idx = face_counter.index(max(face_counter))

        print(f"\nSoggetto principale identificato!")
        print(f"Apparso in {face_counter[idx]} frame su {processed_frames} analizzati")
        print(f"Totale frame salvati: {len(face_frames[idx])}")  # Numero reale di frame
        print(f"Totale persone rilevate: {len(face_counter)}\n")

        return face_frames[idx], encodings_per_person[idx]  # Ritorna TUTTI i frame

definizione nuovo oggetto utile per verificare se un frame contiene il soggetto individuato con l'oggetto precedente (nuovo modello)

In [5]:
import cv2
from insightface.app import FaceAnalysis
import numpy as np

class VideoFaceFilterInsightFace:

    def __init__(self, main_encodings, tolerance=0.6, det_size=(640, 640), resize_scale=0.5):
        self.main_encodings = main_encodings
        self.tolerance = tolerance
        self.resize_scale = resize_scale
        self.app = FaceAnalysis(
            name='buffalo_l',
            providers=['CUDAExecutionProvider']#, 'CPUExecutionProvider']
        )
        self.app.prepare(ctx_id=0, det_size=det_size)
        self.similarity_threshold = tolerance


    def frame_contains_subject(self, frame):
        # Resize per velocità
        small = cv2.resize(frame, (0, 0), fx=self.resize_scale, fy=self.resize_scale)

        # Detect + Encode con InsightFace
        faces = self.app.get(small)

        if not faces:
            return False

        # Check match
        for face in faces:
            enc = face.embedding  # shape (512,) - già normalizzato!

            # ✅ Cosine similarity diretta (embeddings normalizzati)
            similarities = np.dot(self.main_encodings, enc)  # shape (N,)

            # ✅ Usa MASSIMA similarità (non minima distanza)
            if np.max(similarities) > self.tolerance:
                return True

        return False


    def filter_video(self, video_path, sample_interval=1):
        cap = cv2.VideoCapture(video_path)
        kept_frames = []
        total = 0
        kept = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if total % sample_interval == 0:
                if self.frame_contains_subject(frame):
                    kept_frames.append(frame)
                    kept += 1

            total += 1

        cap.release()

        print(f"Frame totali analizzati: {total}")
        print(f"Frame mantenuti: {kept}")

        return kept_frames



    def frames_to_video(self, frames, output_path, fps=25):
        if len(frames) == 0:
            raise ValueError("No frames")

        h, w, _ = frames[0].shape

        # forza consistenza dimensioni
        frames = [cv2.resize(f, (w, h)) for f in frames]

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # prova H264

        out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

        if not out.isOpened():
            raise RuntimeError("VideoWriter non aperto: codec non supportato")

        for f in frames:
            out.write(f)

        out.release()

main che effettua pulizia con nuovo modello (usando metodologia 2: analisi dei frame e non dei video)

In [ ]:
import cv2
import os
from pathlib import Path
import gc
import shutil

root = "/content/drive/MyDrive/FVAB 2025-2026"

video_list = []

file_path = "/content/lista_video.txt"
with open(file_path, "r") as f:
    for row in f:
        row = row.strip()
        if "Real Life Trial" in row or "Bag of Lies" in row:
          video_list.append(row)

video_list.sort()
print(len(video_list))

subjectNotDetected = []
video_path_list = []


extractor = AutoReferenceExtractorInsightFace(
    det_size=(640, 640),  # usa (800,800) per video molto scarsi
    similarity_threshold=0.6  # abbassa a 0.3 se troppo frammentato
)
# fatti 72 al 12/05 ore 09:00
# fatti 80 al 12/05 ore 10:40
# fatti 178 al 12/05 ore 15:15 (GPU)
# fatti 251 al 13/05 ore 04:34
# fatti 286 al 13/05 ore 16:00 (GPU)
# fatti 332 al 14/05 ore 05:16
# fatti 340 al 14/05 ore 09:23 (GPU)
# fatti 438 al 18/05 ore 02:50 (GPU)
for video in video_list[438:]:
      print(f"Sto analizzando {video}")
      reference_image, reference_encodings = extractor.extract_main_face_from_video(video, resize_scale=0.25)

      if (reference_image is None):
            subjectNotDetected.append(video)
            continue

      parts = Path(video).parts

      num = None
      if "Real Life Trial" in video:
            num = parts[9].split('.')[0]
            num = int(num.split('_')[-1])
            frames_root = Path(video).parent.parent.parent / f"extracted_frames/{parts[8]}/{num}"
      else:
            frames_root =  Path(video).parent.parent.parent.parent / f"extracted_frames/{parts[8]}/{parts[9]}"

      print(frames_root)


      if "Real Life Trial" in video:
            dataset_name = "REAL_LIFE_TRIAL"
      else:
            dataset_name = "BAG_OF_LIES"

      if "Truth" in str(video) or "Truthful" in str(video):
            word = f"TRUTHS/{num}"
      elif "Deceptive" in str(video):
            word = f"LIES/{num}"
      else:
            # trova la prima parte che inizia con "User"
            idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
            word = parts[idx] + "/" + parts[idx+1]


      filter = VideoFaceFilterInsightFace(reference_encodings)

      output_path = None

      for path, dirs, files in os.walk(frames_root):
            for file in files:
                  if not file.lower().endswith(".jpg"):
                        continue

                  frame_path = os.path.join(path, file)
                  frame = cv2.imread(frame_path)

                  if frame is None:
                        continue

                  if filter.frame_contains_subject(frame):
                        output_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{os.path.basename(Path(frame_path))}")
                        output_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(output_path), frame)
                        #print(output_path)

                  if output_path is not None and "00.jpg" in output_path.name:
                        print(f"Salvato frame {os.path.basename(output_path)}\n\n")

      if output_path is not None:
            print("Salvataggio dei frame terminato...")
            video_path_list.append(str(output_path.parent))
            print(f"\nVideo finora salvati:")
            for video_path in video_path_list:
                  print(f"{video_path}")
            print("")

      del filter

print("Nessun soggetto nei seguenti video:")
print(subjectNotDetected)

# Salva la lista in un file di testo standard
with open("soggetto_assente_new_model.txt", "w", encoding="utf-8") as f:
      for video in subjectNotDetected:
            f.write(f"{video}\n")

print(f"Salvataggio completato: {len(subjectNotDetected)} video scritti nel file.")

446
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:08<00:00, 31557.15KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
Sto analizza

analisi dei video originali dei dataset coinvolti dalla pulizia con il nuovo modello

In [ ]:
import cv2
import os
from pathlib import Path
import gc


root = "/content/drive/MyDrive/FVAB 2025-2026"

video_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if file.endswith(".mp4") and ("Real Life Trial" in path or "Bag of Lies" in path):
                  video_path = os.path.join(path, file)
                  video = cv2.VideoCapture(video_path)
                  nFrame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                  fps = int(video.get(cv2.CAP_PROP_FPS))
                  video.release()
                  video_list.append((video_path, nFrame, fps))


# Scrittura
with open("original_info2.txt", "w") as f:
      for v in video_list:
            f.write(f"{v[0]},{v[1]},{v[2]}\n")

# Lettura
video_list = []
with open("original_info2.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))


for v in video_list:
      print(v)

('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_0/video.mp4', 245, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_2/video.mp4', 231, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_1/video.mp4', 211, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_3/video.mp4', 210, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_4/video.mp4', 167, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_0/run_5/video.mp4', 220, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_0/video.mp4', 304, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_1/video.mp4', 234, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of Lies/Finalized/User_1/run_2/video.mp4', 218, 30)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Bag of

conversione da frame a video

In [ ]:
import cv2
import os
from glob import glob
from pathlib import Path
import re

root = "/content/drive/MyDrive/DATASET_FVAB"


# Lettura
video_list = []
with open("original_info2.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))

frame_path_list = []

for path, dirs, files in os.walk(root):
      if any(file.lower().endswith('.jpg') for file in files):
            frame_path_list.append(path)


# ordina lessicograficamente
frame_path_list = sorted(frame_path_list)
for p in frame_path_list:
      images = sorted(glob(os.path.join(p, "*.jpg")))

      # Lettura primo frame per ottenere le dimensioni
      frame = cv2.imread(images[0])
      height, width, _ = frame.shape

      fourcc = cv2.VideoWriter_fourcc(*"mp4v")

      video_path = ""
      fps = None
      parts = Path(p).parts

      if "DOLOS" in p:
            continue

      if "BAG_OF_LIES" in p:
            video_path = f"{p}/video.mp4"

            for v in video_list:
                  if f"{parts[-2]}/{parts[-1]}" in v[0]:
                        fps = v[2]
                        break
      else:
            parts = Path(p).parts
            video_path = f"{Path(p).parent}/{parts[-1]}.mp4"
            for v in video_list:
                  if f"{parts[-1]}" in v[0]:
                        if ("TRUTHS" in p and "Truthful" in v[0]) or ("LIES" in p and "Deceptive" in v[0]):
                              fps = v[2]
                              break


      video = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

      for img_path in images:
            frame = cv2.imread(img_path)
            if frame is None:
                  continue
            video.write(frame)

      video.release()
      print(f"Salvato video in {video_path}")

Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_0/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_1/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_2/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_3/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_4/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_5/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_0/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_1/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_2/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_3/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_4/video.mp4
Salvato video in /content/drive/

derivazione frame mantenuti per video da quelli interessati dalla pulizia (dataset bag of lies e real life trial)

In [ ]:
import cv2
import os
from pathlib import Path
import gc


root = "/content/drive/MyDrive/DATASET_FVAB"

video_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if file.endswith(".mp4") and ("REAL_LIFE_TRIAL" in path or "BAG_OF_LIES" in path):
                  video_path = os.path.join(path, file)
                  video = cv2.VideoCapture(video_path)
                  nFrame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                  fps = int(video.get(cv2.CAP_PROP_FPS))
                  video.release()
                  video_list.append((video_path, nFrame, fps))


# Scrittura
with open("cleaned_info2.txt", "w") as f:
      for v in video_list:
            f.write(f"{v[0]},{v[1]},{v[2]}\n")

# Lettura
video_list = []
with open("cleaned_info2.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))


for v in video_list:
      print(v)



('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/1.mp4', 510, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/2.mp4', 1874, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/5.mp4', 1570, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/6.mp4', 546, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/7.mp4', 1403, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/9.mp4', 619, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/10.mp4', 876, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/11.mp4', 1010, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/13.mp4', 571, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/14.mp4', 415, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/15.mp4', 1031, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/16.mp4', 1108, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/LIES/17.mp4', 1315, 29

tiriamo le somme per questo secondo approccio

In [ ]:
import cv2
import os
from pathlib import Path
import gc

# Lettura
original_video_list = []
with open("original_info2.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            original_video_list.append((path, int(nFrame), int(fps)))


# Lettura
cleaned_video_list = []
with open("cleaned_info2.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            cleaned_video_list.append((path, int(nFrame), int(fps)))

print(f"Video totali: {len(original_video_list)}")
print(f"Video puliti: {len(cleaned_video_list)}")


original_video_list_changed = []

ctr_bof = 0
ctr_rlt = 0
for v in original_video_list:

      dataset_name = None
      word = None
      parts = Path(v[0]).parts
      filename = parts[-1]

      if "Real Life Trial" in v[0]:
            dataset_name = "REAL_LIFE_TRIAL"
            ctr_rlt += 1
      else:
            dataset_name = "BAG_OF_LIES"
            ctr_bof += 1


      if "Truth" in str(v[0]) or "Truthful" in str(v[0]):
            word = "TRUTHS"
      elif "Deceptive" in str(v[0]) or "/Lies" in str(v[0]): # necessario "/Lies" perché Lies presente anche in "Bag of Lies"
            word = "LIES"
      else:
            # trova la prima parte che inizia con "User"
            idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
            word = parts[idx] + "/" + parts[idx+1]


      if "Real Life" in str(v[0]):
            filename = f"{int(parts[-1].split("_")[2].split(".")[0])}.mp4"

      # se path non esiste lo crei e poi salvi
      new_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{filename}")


      record = (str(new_path), v[1], v[2])
      original_video_list_changed.append(record)


not_analyzed_list = []
less_frames_list = []
for o in original_video_list_changed:
      flag = False
      for c in cleaned_video_list:
            if o[0] == c[0]:
                  flag = True

                  if c[1] < o[1]:
                        less_frames_list.append(o)
                        less_frames_list.append(c)

      if not flag:
            not_analyzed_list.append(o)

counter_bof = 0
counter_rlt = 0
print(f"\nVideo non analizzati: {len(not_analyzed_list)}")
for v in not_analyzed_list:
      print(v)
      if "BAG_OF_LIES" in v[0]:
            counter_bof += 1
      elif "REAL_LIFE_TRIAL" in v[0]:
            counter_rlt += 1

print(f"\nVideo non analizzati di Bag of Lies: {counter_bof} su {ctr_bof}\nVideo non analizzati di Real Life Trial: {counter_rlt} su {ctr_rlt}")

print(f"\nVideo che hanno subito riduzione di frame: {int(len(less_frames_list)/2)}")
counter = 0
for v in less_frames_list:
      if counter % 2 == 0:
            print("")
      print(v)
      counter += 1 % 2


Video totali: 446
Video puliti: 313

Video non analizzati: 133
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_0/video.mp4', 276, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_1/video.mp4', 242, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_10/video.mp4', 225, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_2/video.mp4', 309, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6/run_8/video.mp4', 224, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7/run_0/video.mp4', 327, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7/run_1/video.mp4', 277, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7/run_4/video.mp4', 336, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8/run_8/video.mp4', 242, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8/run_2/video.mp4', 308, 30)
('/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8/run_3/video.mp4', 315, 30)
('/content/d

nuova pulizia dei frame dal file not_analyzed.txt

In [6]:
import cv2
import os
from pathlib import Path
import gc
import shutil

root = "/content/drive/MyDrive/FVAB 2025-2026"

video_list = []

file_path = "/content/not_analyzed.txt"
with open(file_path, "r") as f:
    for row in f:
        path, _, _ = row.strip().split(",")
        path = path[2:-1].replace("DATASET_FVAB/BAG_OF_LIES", "FVAB 2025-2026/Datasets/Bag of Lies/Finalized")
        video_list.append(path)

video_list.sort()
print(len(video_list))

subjectNotDetected = []
video_path_list = []


extractor = AutoReferenceExtractorInsightFace(
    det_size=(320, 320),  # usa (800,800) per video molto scarsi
    similarity_threshold=0.6  # abbassa a 0.3 se troppo frammentato
)

for video in video_list[68:]:
      print(f"Sto analizzando {video}")
      reference_image, reference_encodings = extractor.extract_main_face_from_video(video, resize_scale=0.25)

      if (reference_image is None):
            subjectNotDetected.append(str(video))
            continue

      parts = Path(video).parts

      num = None
      if "Real Life Trial" in video:
            num = parts[9].split('.')[0]
            num = int(num.split('_')[-1])
            frames_root = Path(video).parent.parent.parent / f"extracted_frames/{parts[8]}/{num}"
      else:
            frames_root =  Path(video).parent.parent.parent.parent / f"extracted_frames/{parts[8]}/{parts[9]}"

      print(frames_root)


      if "Real Life Trial" in video:
            dataset_name = "REAL_LIFE_TRIAL"
      else:
            dataset_name = "BAG_OF_LIES"

      if "Truth" in str(video) or "Truthful" in str(video):
            word = f"TRUTHS/{num}"
      elif "Deceptive" in str(video):
            word = f"LIES/{num}"
      else:
            # trova la prima parte che inizia con "User"
            idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
            word = parts[idx] + "/" + parts[idx+1]


      filter = VideoFaceFilterInsightFace(main_encodings = reference_encodings, det_size=(320, 320))

      output_path = None

      for path, dirs, files in os.walk(frames_root):
            for file in files:
                  if not file.lower().endswith(".jpg"):
                        continue

                  frame_path = os.path.join(path, file)
                  frame = cv2.imread(frame_path)

                  if frame is None:
                        continue

                  if filter.frame_contains_subject(frame):
                        output_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{os.path.basename(Path(frame_path))}")
                        output_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(output_path), frame)
                        #print(output_path)

                  if output_path is not None and "00.jpg" in output_path.name:
                        print(f"Salvato frame {os.path.basename(output_path)}\n\n")

      if output_path is not None:
            print("Salvataggio dei frame terminato...")
            video_path_list.append(str(output_path.parent))
            print(f"\nVideo finora salvati:")
            for video_path in video_path_list:
                  print(f"{video_path}")
            print("")

      del filter

print("Nessun soggetto nei seguenti video:")
print(subjectNotDetected)

# Salva la lista in un file di testo standard
with open("soggetto_assente.txt", "w", encoding="utf-8") as f:
      for video in subjectNotDetected:
            f.write(f"{video}\n")

print(f"Salvataggio completato: {len(subjectNotDetected)} video scritti nel file.")

133
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:04<00:00, 64733.00KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (320, 320)
Sto analizza

conversione da frame a video

In [4]:
import cv2
import os
from glob import glob
from pathlib import Path
import re

root = "/content/drive/MyDrive/DATASET_FVAB"


# Lettura
video_list = []
frame_path_list = []
with open("not_analyzed.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip("()\n").split(",")
            video_path = path[1:-1].replace("DATASET_FVAB/BAG_OF_LIES", "FVAB 2025-2026/Datasets/Bag of Lies/Finalized")
            video_list.append((video_path, int(nFrame), int(fps)))

            frame_path = path[1:-1].replace("/video.mp4", "")
            frame_path_list.append(frame_path)
            #print(frame_path_list[-1])

# ordina lessicograficamente
frame_path_list = sorted(frame_path_list)
for p in frame_path_list:
      images = sorted(glob(os.path.join(p, "*.jpg")))

      # Lettura primo frame per ottenere le dimensioni
      frame = cv2.imread(images[0])
      height, width, _ = frame.shape

      fourcc = cv2.VideoWriter_fourcc(*"mp4v")

      video_path = ""
      fps = None
      parts = Path(p).parts

      if "DOLOS" in p:
            continue

      if "BAG_OF_LIES" in p:
            video_path = f"{p}/video.mp4"

            for v in video_list:
                  if f"{parts[-2]}/{parts[-1]}" in v[0]:
                        fps = v[2]
                        break
      else:
            parts = Path(p).parts
            video_path = f"{Path(p).parent}/{parts[-1]}.mp4"
            for v in video_list:
                  if f"{parts[-1]}" in v[0]:
                        if ("TRUTHS" in p and "Truthful" in v[0]) or ("LIES" in p and "Deceptive" in v[0]):
                              fps = v[2]
                              break


      video = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

      for img_path in images:
            frame = cv2.imread(img_path)
            if frame is None:
                  continue
            video.write(frame)

      video.release()
      print(f"Salvato video in {video_path}")

Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_1/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_2/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_3/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_4/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_5/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_6/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10/run_7/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_11/run_1/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_11/run_2/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_11/run_3/video.mp4
Salvato video in /content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_11/run_4/video.mp4
Salvato video in /con

TIRIAMO LE SOMME PER LA TERZA VOLTA (NON CE LA FACCIO PIU')

In [5]:
import cv2
import os
from pathlib import Path
import gc


root = "/content/drive/MyDrive/FVAB 2025-2026"

video_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if file.endswith(".mp4"):
                  video_path = os.path.join(path, file)
                  video = cv2.VideoCapture(video_path)
                  nFrame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                  fps = int(video.get(cv2.CAP_PROP_FPS))
                  video.release()
                  video_list.append((video_path, nFrame, fps))


# Scrittura
with open("original_info3.txt", "w") as f:
      for v in video_list:
            f.write(f"{v[0]},{v[1]},{v[2]}\n")

# Lettura
video_list = []
with open("original_info3.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))


for v in video_list:
      print(v)

('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_004.mp4', 2443, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_001.mp4', 427, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_016.mp4', 188, 25)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_012.mp4', 856, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_015.mp4', 1109, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_006.mp4', 918, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_010.mp4', 2188, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_011.mp4', 1278, 29)
('/content/drive/MyDrive/FVAB 2025-2026/Datasets/Real Life Trial/Clips/Truthful/trial_truth_007.mp4', 2147, 

In [6]:
import cv2
import os
from pathlib import Path
import gc


root = "/content/drive/MyDrive/DATASET_FVAB"

video_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if file.endswith(".mp4"):
                  video_path = os.path.join(path, file)
                  video = cv2.VideoCapture(video_path)
                  nFrame = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                  fps = int(video.get(cv2.CAP_PROP_FPS))
                  video.release()
                  video_list.append((video_path, nFrame, fps))


# Scrittura
with open("cleaned_info3.txt", "w") as f:
      for v in video_list:
            f.write(f"{v[0]},{v[1]},{v[2]}\n")

# Lettura
video_list = []
with open("cleaned_info3.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            video_list.append((path, int(nFrame), int(fps)))


for v in video_list:
      print(v)



('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP16_lie4.mp4', 89, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP15_lie16.mp4', 119, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP17_lie17.mp4', 63, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP16_lie7.mp4', 75, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP22_lie28.mp4', 185, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP16_lie19.mp4', 100, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP19_lie14.mp4', 59, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP18_lie9.mp4', 150, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP21_lie9.mp4', 53, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP15_lie5.mp4', 87, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP22_lie5.mp4', 75, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/AN_WILTY_EP24_lie2.mp4', 250, 25)
('/con

In [7]:
import cv2
import os
from pathlib import Path
import gc

# Lettura
original_video_list = []
with open("original_info3.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            original_video_list.append((path, int(nFrame), int(fps)))


# Lettura
cleaned_video_list = []
with open("cleaned_info3.txt", "r") as f:
      for linea in f:
            path, nFrame, fps = linea.strip().split(",")
            cleaned_video_list.append((path, int(nFrame), int(fps)))

print(f"Video totali: {len(original_video_list)}")
print(f"Video puliti: {len(cleaned_video_list)}")


original_video_list_changed = []

ctr_bof = 0
ctr_rlt = 0
ctr_dol = 0
for v in original_video_list:

      dataset_name = None
      word = None
      parts = Path(v[0]).parts
      filename = parts[-1]

      if "Real Life Trial" in v[0]:
            dataset_name = "REAL_LIFE_TRIAL"
            ctr_rlt += 1
      elif "Bag" in v[0]:
            dataset_name = "BAG_OF_LIES"
            ctr_bof += 1
      else:
            dataset_name = "DOLOS"
            ctr_dol += 1


      if "Truth" in str(v[0]) or "Truthful" in str(v[0]):
            word = "TRUTHS"
      elif "Deceptive" in str(v[0]) or "/Lies" in str(v[0]): # necessario "/Lies" perché Lies presente anche in "Bag of Lies"
            word = "LIES"
      else:
            # trova la prima parte che inizia con "User"
            idx = next((i for i, p in enumerate(parts) if p.startswith("User")), None)
            word = parts[idx] + "/" + parts[idx+1]

      if "Real Life" in str(v[0]):
            filename = f"{int(parts[-1].split("_")[2].split(".")[0])}.mp4"

      # se path non esiste lo crei e poi salvi
      new_path = Path(f"/content/drive/MyDrive/DATASET_FVAB/{dataset_name}/{word}/{filename}")


      record = (str(new_path), v[1], v[2])
      original_video_list_changed.append(record)


not_analyzed_list = []
less_frames_list = []
for o in original_video_list_changed:
      flag = False
      for c in cleaned_video_list:
            if o[0] == c[0]:
                  flag = True

                  if c[1] < o[1]:
                        less_frames_list.append(o)
                        less_frames_list.append(c)

      if not flag:
            not_analyzed_list.append(o)

counter_bof = 0
counter_rlt = 0
counter_dol = 0
print(f"\nVideo non analizzati: {len(not_analyzed_list)}")
for v in not_analyzed_list:
      print(v)
      if "BAG_OF_LIES" in v[0]:
            counter_bof += 1
      elif "REAL_LIFE_TRIAL" in v[0]:
            counter_rlt += 1
      else:
            counter_dol += 1

print(f"\nVideo non analizzati di DOLOS: {counter_dol} su {ctr_dol}\nVideo non analizzati di Bag of Lies: {counter_bof} su {ctr_bof}\nVideo non analizzati di Real Life Trial: {counter_rlt} su {ctr_rlt}")

print(f"\nVideo che hanno subito riduzione di frame: {int(len(less_frames_list)/2)}")
counter = 0
for v in less_frames_list:
      if counter % 2 == 0:
            print("")
      print(v)
      counter += 1 % 2


Video totali: 2094
Video puliti: 2092

Video non analizzati: 2
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/SB_WILTY_EP32_lie2.mp4', 125, 25)
('/content/drive/MyDrive/DATASET_FVAB/DOLOS/LIES/SB_WILTY_EP32_lie3.mp4', 125, 25)

Video non analizzati di DOLOS: 2 su 1648
Video non analizzati di Bag of Lies: 0 su 325
Video non analizzati di Real Life Trial: 0 su 121

Video che hanno subito riduzione di frame: 1071

('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/1.mp4', 427, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/1.mp4', 412, 29)

('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/12.mp4', 856, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/12.mp4', 851, 29)

('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/15.mp4', 1109, 29)
('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/15.mp4', 1067, 29)

('/content/drive/MyDrive/DATASET_FVAB/REAL_LIFE_TRIAL/TRUTHS/10.mp4', 2188, 29)
('/content/drive/MyDrive/D

organizzazione per petrillo: dataset Bag of Lies ha per ogni user una cartella TRUTHS e LIES

In [13]:
import os
import shutil
import pandas as pd
from pathlib import Path

# Cartella principale
root = "/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES"

video_path_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if "video" in file:
                  video_path = os.path.join(path, file)
                  video_path_list.append(video_path)
                  #print(video_path)

# Legge il file CSV
df = pd.read_csv("Annotations.csv")
filtered_df = df[["video", "truth"]]

# Mostra le prime righe
#print(df["video"].iloc[0])
#print(df["truth"])

truth_lie_list = []

for video, truth in zip(df["video"], df["truth"]):
      video = video[11:]
      for video_path in video_path_list:
            if video in video_path:
                  file = Path(video_path)
                  run = file.parts[-2]

                  if truth:
                        new_name = file.with_name(f"{run}_truth.mp4")
                  else:
                        new_name = file.with_name(f"{run}_lie.mp4")

                  file.rename(new_name)
                  #print(new_name)
                  break



/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_0/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_2/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_1/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_3/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_4/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0/run_5/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_0/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_1/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_2/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_3/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_5/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1/run_4/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_2/run_0/video.mp4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_2/run_1/video.mp4
/conte

copio i video nella nuova gerarchia

In [25]:
import os
import shutil
import pandas as pd
from pathlib import Path

# Cartella principale
root = "/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES"


# prendo i path dei video di BoF
video_path_list = []
for path, dirs, files in os.walk(root):
      for file in files:
            if ".mp4" in file:
                  video_path = os.path.join(path, file)
                  video_path_list.append(video_path)


# prendo i path degli utenti di BoF
dir_path_list = []
for path, dirs, files in os.walk(root):
      for dir in dirs:
            dir_path = os.path.join(path, dir)
            dir_path_list.append(dir_path)
            #print(dir_path)
      break

# aggiungo cartelle TRUTHS e LIES per ogni User
for dir_path in dir_path_list:
    path = Path(dir_path)
    truth_dir = path / "TRUTHS"
    lie_dir = path / "LIES"
    truth_dir.mkdir(exist_ok=True)
    lie_dir.mkdir(exist_ok=True)

for video_path in video_path_list:
      parts = Path(video_path).parts
      word = None

      if "lie" in video_path:
            word = "LIES"
      elif "truth" in video_path:
            word = "TRUTHS"
      new_path = Path(*parts[:-2])/word/parts[-1]
      shutil.copy(video_path, new_path)
      print(new_path)





/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_0
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_1
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_2
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_3
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_5
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_6
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_11
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_10
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_13
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_12
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_21
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_4
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_7
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_8
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_9
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_14
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_15
/content/drive/MyDrive/DATASET_FVAB/BAG_OF_LIES/User_16
/c